# Week 7 lecture walkthrough: repeated persistence through physical time

This is the live worked example, built around a synthetic ring that fills in and re-forms. It follows the conceptual argument of the slides through prediction, reveal and interpretation. It is not intended as a line-by-line answer key to the participant practical.

**Resource boundary.** The [reference notes](index.qmd) distinguish representation time, physical time and filtration scale. The [slides](slides.qmd) stage that distinction visually. This notebook completes the framewise calculation and CROCKER-style summary. The [participant practical](lab.ipynb) leaves a representation choice and the interpretation boundary to students.

**Lecture map.** Build the state-space representation, inspect selected frames, predict the time course of loop evidence, and then reveal the two-axis summary. End by asking what feature identity would require beyond repeated persistence.

We compute ordinary Rips persistence independently at each frame of a synthetic evolving point cloud. The output is a time series of diagrams and a CROCKER-style rank surface. No feature identity is asserted across frames.

**Presenter route.** Use the ring-to-disk-to-ring sequence to keep physical time separate from filtration scale and to show why repeated diagrams do not track identity.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
from ripser import ripser

def betti_curve(D,grid): return np.array([np.sum((D[:,0]<=a)&(a<D[:,1])) for a in grid])
def max_persistence(D):
 F=D[np.isfinite(D[:,1])]; return float(np.max(F[:,1]-F[:,0])) if len(F) else 0.

## Representation bridge: delay coordinates

A scalar signal is not yet a state-space point cloud. For lag $\tau$ and dimension $m$ construct

$$\Phi_{m,\tau}(t)=\bigl(x(t),x(t-\tau),\ldots,x(t-(m-1)\tau)\bigr).$$

The code below uses index lag rather than physical units. Predict the shape for a periodic signal. Then change only `lag` and describe what geometric organisation changes. Temporal order is not retained by the later Rips calculation unless it is stored separately.

In [ ]:
def delay_coordinates(signal, dimension, lag):
    rows = len(signal) - (dimension - 1) * lag
    return np.column_stack([
        signal[(dimension - 1 - j) * lag:(dimension - 1 - j) * lag + rows]
        for j in range(dimension)
    ])

sample_times = np.linspace(0, 8 * np.pi, 500)
signal = np.sin(sample_times)
lag = 25
embedded = delay_coordinates(signal, dimension=2, lag=lag)
plt.figure(figsize=(4, 4))
plt.plot(embedded[:, 0], embedded[:, 1], linewidth=1)
plt.gca().set_aspect('equal')
plt.xlabel(r'$x(t)$'); plt.ylabel(r'$x(t-\tau)$')
plt.show()
print('embedded shape:', embedded.shape, 'index lag:', lag)

## 1. Observe: lecture demonstration

**Presenter cue.** Show the object before the calculation. Ask the room to separate what is given from what will be constructed.

The synthetic system moves from a coherent ring to a filled cloud and back. The samples at each frame are observations of a changing representation. The point labels are reused only to generate smooth motion; persistence does not use those labels.

In [ ]:
n=70; times=np.linspace(0,1,17); theta=np.linspace(0,2*np.pi,n,endpoint=False)
base_angle=theta+.03*RNG.normal(size=n); target_radius=np.sqrt(RNG.random(n))
frames=[]
for t in times:
 mix=np.sin(np.pi*t)**2
 radius=(1-mix)*np.ones(n)+mix*target_radius
 P=np.c_[radius*np.cos(base_angle),radius*np.sin(base_angle)]+.025*RNG.normal(size=(n,2))
 frames.append(P)
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,i in zip(axes,[0,8,16]): ax.scatter(*frames[i].T,s=12); ax.set_title(f't={times[i]:.2f}'); ax.set_aspect('equal')
plt.show()

## 2. Predict: lecture demonstration

**Presenter cue.** Pause here and collect at least two predictions before revealing any output.

1. When should the longest $H_1$ interval be largest?
2. At a fixed Rips threshold, when should $\beta_1$ be nonzero?
3. Does a long bar at consecutive frames establish that it is the same feature?
4. How might window length change the scientific interpretation?

## 3. Implement: lecture demonstration

**Reveal.** Run one cell at a time. Name the domain, codomain, complex, module or summary before interpreting its values.

Compute a separate diagram at every physical time. Keep physical time $t$ distinct from filtration threshold $\varepsilon$.

In [ ]:
D=[ripser(P,maxdim=1)['dgms'][1] for P in frames]
maxpers=np.array([max_persistence(d) for d in D])
plt.plot(times,maxpers,marker='o'); plt.xlabel('physical time t'); plt.ylabel('maximum H1 persistence'); plt.show()

## 4. Compare: lecture demonstration

**Controlled comparison.** Keep the stated input fixed and change only the highlighted modelling decision.

Construct a CROCKER-style surface $\beta_1(t,\varepsilon)$: each cell counts intervals alive at physical time $t$ and Rips threshold $\varepsilon$. Compare it with the simpler radial coefficient-of-variation baseline.

In [ ]:
grid=np.linspace(0,1.8,150)
B=np.vstack([betti_curve(d,grid) for d in D])
radial_cv=np.array([np.linalg.norm(P-P.mean(0),axis=1).std()/np.linalg.norm(P-P.mean(0),axis=1).mean() for P in frames])
fig,axes=plt.subplots(2,1,figsize=(8,5),sharex=True)
im=axes[0].imshow(B.T,origin='lower',aspect='auto',extent=[times[0],times[-1],grid[0],grid[-1]])
axes[0].set_ylabel('Rips threshold ε'); axes[0].set_title('CROCKER-style beta_1(t, ε)'); fig.colorbar(im,ax=axes[0],label='beta_1')
axes[1].plot(times,radial_cv); axes[1].set_xlabel('physical time t'); axes[1].set_ylabel('radial CV'); plt.show()

## 5. Interpret: lecture demonstration

**Presenter close.** Ask what the example supports, what information was discarded and which stronger claim would be unjustified.

1. Which axis is physical time and which is filtration scale?
2. What statement can the rank surface support without feature tracking?
3. When would a sliding window blur a transition or create apparent persistence?
4. Does topology add information beyond radial variation in this synthetic example?
5. What extra maps or correspondences would be required to claim feature identity?

**† Qualification.** A smooth time series of summaries is still repeated static persistence. It is not a vineyard or vineyard module.

The rank surface supports statements about how the distribution of topological lifetimes changes with time and scale. It does not label a class across frames. In this designed ring-to-disk example, radial variation is already a strong baseline, so the demonstration motivates method comparison rather than superiority.

## Lecture close

Return to the final slide questions.

1. Name the observed or starting object.
2. Name every constructed object used in this walkthrough.
3. Identify the single modelling decision that drove the central comparison.
4. State one conclusion supported by the calculation and one conclusion it cannot establish.

**Take-forward example.** The purpose of a synthetic ring that fills in and re-forms is to make repeated persistence through physical time concrete. The example is deliberately small or synthetic so that the construction remains inspectable.